# AI Crypto Hedge Fund - Final Notebook

Execution mode: **FULL FINAL NOTEBOOK**.

This is the single end-to-end reviewer narrative for the public research MVP. It imports repository package code, validates visible methodology evidence, and reads committed validation/final-test artifacts. It does **not** place orders, download live data, call an external LLM, rerun `make final-test`, or retune exposed final-test results.

Output labels used below:

- **RECOMPUTED SAFELY FROM INCLUDED PRE-2025 DATA**: deterministic checks that use train/validation-period data only.
- **LOADED FROM FROZEN 2025 FINAL-TEST ARTIFACTS - NOT RECOMPUTED**: evidence loaded from the accepted exposed final-test directory.


In [1]:
import json
from pathlib import Path

import pandas as pd

from crypto_hedge_fund.features.level2 import LEVEL2_FEATURE_COLUMNS, build_level2_feature_frame
from crypto_hedge_fund.features.level5 import Level5ScoringConfig
from crypto_hedge_fund.models.ml import _pipeline
from crypto_hedge_fund.reporting import load_stage12_context
from crypto_hedge_fund.reporting.context import (
    format_float,
    markdown_table,
    representative_trace_rows,
)

ctx = load_stage12_context(Path("."))
FINAL_DIR = Path("artifacts/final_test/dab407601cba")


def pct(value, digits=2):
    return f"{float(value) * 100:.{digits}f}%"


def money(value):
    return f"${float(value):,.0f}"


print("final_test_lock_sha256:", ctx.lock_hash)
print("final_test_exposure:", ctx.suite_summary["final_test_exposure"])
print("final_test_dir:", str(FINAL_DIR))
print("stage14_scope:", "reviewer-facing notebook/report/deck transparency only")
print("no_live_trading:", True)

final_test_lock_sha256: dab407601cbaf8198361e5e3d074260546ed4bbab4c4be2555248b246631308b
final_test_exposure: EXPOSED
final_test_dir: artifacts/final_test/dab407601cba
stage14_scope: reviewer-facing notebook/report/deck transparency only
no_live_trading: True


## 1. Scope, reproducibility and quarantine

This repository is an educational historical research system, not investment advice and not an enabled live trading bot. The MVP is long-only, unlevered, daily-bar spot with USDT cash. The final-test suite is already exposed, so this notebook surfaces existing evidence without changing methodology.


In [2]:
summary = ctx.suite_summary
for key in [
    "data_sha256",
    "instruments_sha256",
    "manifest_sha256",
    "validation_selected_sha256",
    "generated_final_config_sha256",
    "locked_git_commit",
    "git_commit",
]:
    print(f"{key}: {summary[key]}")
print("period:", summary["period"])
print("costs:", summary["cost_assumptions"])
print("runner_source_dirty_stage11:", ctx.traces["level_5"]["provenance"]["git_worktree_dirty"])

data_sha256: 9f539f38394240f5dcd922b23d234008a84a357c38ef9f2d8197acfab80d7e14
instruments_sha256: df7777139dd4106032280339818ba18179882c8e19141f374d87cb8e7bddf18b
manifest_sha256: 24b2263f9ffd125fa06811fe689960b8bd5bdabce9f5e933bd0d0c4b37fcbd3e
validation_selected_sha256: 3f2dd08bbec595d6233852bfc94de6eae0a2cdb91d6aeec1f408afbbd10046cf
generated_final_config_sha256: c6c79b974e7c46f4a01781fb8e2b1a96304e1c3f639a10f38fd9a0d2b1522fc6
locked_git_commit: 394d146523445ed53c978ade1033cc7870237a8f
git_commit: 6aad82116feb26f83b7658414207e03371e07864
period: ['2025-01-01', '2025-12-31']
costs: {'chargeable_notional': 'risky_asset_notional_traded_only', 'fee_bps_one_way': 10.0, 'initial_capital_usd': 1000000, 'risk_free_rate': 0.0, 'slippage_bps_one_way': 5.0}
runner_source_dirty_stage11: True


## 2. Data clock and executable Level 2 target

**RECOMPUTED SAFELY FROM INCLUDED PRE-2025 DATA.** Daily bars are UTC bar-start records. A completed source bar at `t` creates features after bar close; decisions execute at the next open `t+1`; the supervised Level 2 label evaluates the following open-to-open move ending at `t+2`.

`label_t = 1[open(t+2) / open(t+1) - 1 > threshold_return]`, where `threshold_return = (fee_bps + slippage_bps + safety_margin_bps) / 10000 = 0.0020`.


In [3]:
bars = pd.read_parquet(
    "data/processed/ohlcv_daily.parquet",
    columns=[
        "bar_start_utc",
        "bar_end_utc",
        "symbol",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "dollar_volume",
        "exchange",
        "market_type",
        "timeframe",
    ],
)
bars["bar_start_utc"] = pd.to_datetime(bars["bar_start_utc"], utc=True)
pre_2025_btc = bars.loc[
    (bars["symbol"] == "BTC/USDT") & (bars["bar_start_utc"] <= pd.Timestamp("2024-12-31", tz="UTC"))
].copy()
features = build_level2_feature_frame(
    pre_2025_btc,
    symbol="BTC/USDT",
    horizon_open_days=1,
    threshold_return=0.0020,
)
print("feature_rows_pre_2025:", len(features))
print("feature_columns_count:", len(LEVEL2_FEATURE_COLUMNS))
print("feature_columns:", ", ".join(LEVEL2_FEATURE_COLUMNS))
print("target_threshold_return:", features["target_threshold_return"].iloc[0])
print("sample_timing:")
print(
    features[
        [
            "bar_start_utc",
            "feature_cutoff",
            "execution_time",
            "future_open_time",
            "forward_return",
            "target_label",
        ]
    ]
    .tail(3)
    .to_string(index=False)
)

feature_rows_pre_2025: 1399
feature_columns_count: 20
feature_columns: open_return_1d, close_return_1d, return_5d, return_10d, return_20d, sma_ratio_10_50, ema_ratio_12_26, rsi_14, macd, macd_signal, atr_14_norm, realized_vol_7, realized_vol_20, realized_vol_60, range_norm, close_open_return, gap_return, drawdown_60, volume_z_20, dollar_volume_z_20
target_threshold_return: 0.002
sample_timing:
            bar_start_utc            feature_cutoff            execution_time          future_open_time  forward_return  target_label
2024-12-27 00:00:00+00:00 2024-12-28 00:00:00+00:00 2024-12-28 00:00:00+00:00 2024-12-29 00:00:00+00:00        0.010615             1
2024-12-28 00:00:00+00:00 2024-12-29 00:00:00+00:00 2024-12-29 00:00:00+00:00 2024-12-30 00:00:00+00:00       -0.016388             0
2024-12-29 00:00:00+00:00 2024-12-30 00:00:00+00:00 2024-12-30 00:00:00+00:00 2024-12-31 00:00:00+00:00       -0.010093             0


## 3. Exact Level 2 model and agent specifications

The single-pair Level 2 experiment compares deterministic SMA, econometric AR/GARCH, two classical ML classifiers, and a deterministic typed-agent ensemble under the same next-open execution, cost model and ledger.


In [4]:
model_rows = [
    {
        "component": "technical_sma",
        "class_or_rule": "deterministic SMA crossover",
        "configuration": "Level 2 fast=10, slow=50; no supervised target",
    },
    {
        "component": "econometric_ar_garch",
        "class_or_rule": "statsmodels.tsa.ar_model.AutoReg + arch.arch_model GARCH(1,1)",
        "configuration": (
            "daily_causal expanding refit; deterministic educational GARCH fallback "
            "if arch fit is unavailable"
        ),
    },
    {
        "component": "ml_logistic",
        "class_or_rule": "sklearn.linear_model.LogisticRegression",
        "configuration": str(_pipeline("ml_logistic", seed=7)),
    },
    {
        "component": "ml_hist_gradient_boosting",
        "class_or_rule": "sklearn.ensemble.HistGradientBoostingClassifier",
        "configuration": str(_pipeline("ml_hist_gradient_boosting", seed=7)),
    },
    {
        "component": "agent_ensemble",
        "class_or_rule": "deterministic aggregation over typed agent outputs",
        "configuration": (
            "component weights 0.25 each; confidence-weighted over active "
            "non-abstaining agents; invalid/stale/non-finite outputs -> zero "
            "confidence/reason codes"
        ),
    },
]
print(markdown_table(model_rows, ["component", "class_or_rule", "configuration"]))
print("Level5ScoringConfig:", Level5ScoringConfig())

| component | class_or_rule | configuration |
| --- | --- | --- |
| technical_sma | deterministic SMA crossover | Level 2 fast=10, slow=50; no supervised target |
| econometric_ar_garch | statsmodels.tsa.ar_model.AutoReg + arch.arch_model GARCH(1,1) | daily_causal expanding refit; deterministic educational GARCH fallback if arch fit is unavailable |
| ml_logistic | sklearn.linear_model.LogisticRegression | Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=500,
                                    random_state=7))]) |
| ml_hist_gradient_boosting | sklearn.ensemble.HistGradientBoostingClassifier | Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('model',
                 HistGradientBoostingClassifier(l2_regularization=0.01,
                                                learning_rate=0.05, max_iter=60,
 

## 4. Training cadence, predictions and leakage audit

**LOADED FROM COMMITTED VALIDATION ARTIFACTS.** ML classifiers use monthly expanding walk-forward fits. The econometric agent uses `daily_causal` expanding refits. Available labels must satisfy `label_observation_time < first_execution`; the fit audit reports zero future-label flags.


In [5]:
fit_audit = pd.read_parquet("artifacts/monitoring/level_2_fit_audit.parquet")
preds = pd.read_parquet("artifacts/monitoring/level_2_model_predictions.parquet")
summary_rows = []
for model, frame in fit_audit.groupby("model"):
    summary_rows.append(
        {
            "model": model,
            "rows": len(frame),
            "refit_frequency": ",".join(sorted(frame["refit_frequency"].dropna().unique())),
            "unique_fit_cutoffs": frame["fit_cutoff"].nunique(),
            "future_label_flags": int(frame["used_future_labels"].sum()),
            "train_samples_min": int(frame["train_samples"].min()),
            "train_samples_max": int(frame["train_samples"].max()),
        }
    )
print(
    markdown_table(
        summary_rows,
        [
            "model",
            "rows",
            "refit_frequency",
            "unique_fit_cutoffs",
            "future_label_flags",
            "train_samples_min",
            "train_samples_max",
        ],
    )
)
print("prediction_rows:", len(preds))
print("prediction_approaches:", preds["approach"].value_counts().to_dict())

| model | rows | refit_frequency | unique_fit_cutoffs | future_label_flags | train_samples_min | train_samples_max |
| --- | --- | --- | --- | --- | --- | --- |
| econometric_ar_garch | 365 | daily_causal | 365 | 0 | 1033 | 1397 |
| ml_hist_gradient_boosting | 1095 | monthly | 12 | 0 | 1033 | 1368 |
| ml_logistic | 1095 | monthly | 12 | 0 | 1033 | 1368 |
prediction_rows: 1095
prediction_approaches: {'econometric_ar_garch': 365, 'ml_logistic': 365, 'ml_hist_gradient_boosting': 365}


## 5. Validation selection before final-test evidence

**LOADED FROM VALIDATION ARTIFACTS.** The final-test period was not used to choose methods. Level 2 `agent_ensemble` is shown honestly as the representative multi-agent submission, not as the numerical validation-return or validation-Sharpe winner.


In [6]:
def validation_rows(level):
    frame = ctx.validation_metrics[level]
    label_col = next((c for c in ["approach", "method", "policy"] if c in frame.columns), "level")
    selected_col = f"selected_for_{level}"
    rows = []
    for _, row in frame.iterrows():
        rows.append(
            {
                "candidate": row.get(label_col, level),
                "selected": bool(row.get(selected_col, False)),
                "net_return": pct(row["net_total_return"]),
                "sharpe": format_float(row["net_sharpe"]),
                "mdd": pct(row["net_max_drawdown"]),
                "turnover": format_float(row.get("net_turnover", 0.0)),
                "cost": money(row.get("net_total_cost", 0.0)),
            }
        )
    return rows


for level in ["level_1", "level_2", "level_3", "level_4", "level_5"]:
    print("\n" + level.upper())
    print(
        markdown_table(
            validation_rows(level),
            ["candidate", "selected", "net_return", "sharpe", "mdd", "turnover", "cost"],
        )
    )
print("\nselection_notes:")
print(
    "- Level 2 ensemble frozen as representative multi-agent candidate, "
    "not max validation Sharpe/return."
)
print("- Level 3 cvar_downside selected by validation Sharpe under constraints.")
print(
    "- Level 4 calendar_monthly selected because drift/signal alternatives "
    "violated turnover constraints."
)
print("- Level 5 is deterministic cross-sectional scoring, not a fitted classifier/regressor.")


LEVEL_1
| candidate | selected | net_return | sharpe | mdd | turnover | cost |
| --- | --- | --- | --- | --- | --- | --- |
| level_1 | False | 118.61% | 1.95 | -20.03% | 5.01 | $10,771 |

LEVEL_2
| candidate | selected | net_return | sharpe | mdd | turnover | cost |
| --- | --- | --- | --- | --- | --- | --- |
| technical_sma | False | 45.25% | 1.13 | -34.33% | 13.38 | $23,529 |
| econometric_ar_garch | False | 2.72% | 0.59 | -3.38% | 24.18 | $37,110 |
| ml_logistic | False | 1.41% | 1.42 | -0.28% | 2.37 | $3,591 |
| ml_hist_gradient_boosting | False | 2.84% | 1.35 | -1.04% | 5.77 | $8,800 |
| agent_ensemble | True | 2.82% | 1.11 | -2.30% | 2.64 | $4,019 |

LEVEL_3
| candidate | selected | net_return | sharpe | mdd | turnover | cost |
| --- | --- | --- | --- | --- | --- | --- |
| equal_weight | False | 128.15% | 1.68 | -36.33% | 0.99 | $1,492 |
| inverse_volatility | False | 121.84% | 1.67 | -35.16% | 0.99 | $1,492 |
| minimum_variance | False | 111.35% | 1.65 | -32.80% | 0.99 | $1,492

## 6. Robustness and predictive diagnostics

**LOADED FROM VALIDATION ARTIFACTS.** Robustness checks are reported as uncertainty evidence, not as proof of alpha. Predictive metrics are separate from after-cost trading metrics.


In [7]:
robustness = json.loads(Path("artifacts/monitoring/level_2_robustness.json").read_text())[
    "robustness"
]
bootstrap = robustness["block_bootstrap"]
randomization = robustness["circular_shift_randomization"]
print("moving_block_bootstrap_repetitions:", bootstrap["repetitions"])
print("block_length_days:", bootstrap["block_length_days"])
print("selected_validation_sharpe_ci_95:", bootstrap["sharpe_ci_95"])
print("circular_shift_repetitions:", randomization["repetitions"])
print(
    "observed_score_forward_return_correlation:",
    round(randomization["observed_score_forward_return_correlation"], 4),
)
print("two_sided_p_value:", round(randomization["two_sided_p_value"], 4))
print("multiple_seeds:", robustness["multiple_seeds"]["seeds"])
print("interpretation:", "statistically inconclusive; no robust alpha claim")

l2_final = ctx.metrics["level_2"]
pred_rows = []
for _, row in l2_final.iterrows():
    approach = row["approach"]
    if approach.startswith("ml_"):
        pred_rows.append(
            {
                "approach": approach,
                "log_loss": format_float(row.get("predictive_log_loss"), 3),
                "brier": format_float(row.get("predictive_brier_score"), 3),
                "roc_auc": format_float(row.get("predictive_roc_auc"), 3),
                "pr_auc": format_float(row.get("predictive_pr_auc"), 3),
                "net_return": pct(row["net_total_return"]),
                "net_sharpe": format_float(row["net_sharpe"]),
            }
        )
print("\nFinal-test ML predictive metrics loaded as evidence only:")
print(
    markdown_table(
        pred_rows,
        ["approach", "log_loss", "brier", "roc_auc", "pr_auc", "net_return", "net_sharpe"],
    )
)

moving_block_bootstrap_repetitions: 1000
block_length_days: 14
selected_validation_sharpe_ci_95: [-1.0494482228176896, 3.020091761696224]
circular_shift_repetitions: 1000
observed_score_forward_return_correlation: -0.0177
two_sided_p_value: 0.6823
multiple_seeds: [7, 42, 137]
interpretation: statistically inconclusive; no robust alpha claim

Final-test ML predictive metrics loaded as evidence only:
| approach | log_loss | brier | roc_auc | pr_auc | net_return | net_sharpe |
| --- | --- | --- | --- | --- | --- | --- |
| ml_logistic | 0.689 | 0.248 | 0.515 | 0.501 | -0.36% | -0.60 |
| ml_hist_gradient_boosting | 0.706 | 0.256 | 0.497 | 0.453 | 0.02% | 0.02 |


## 7. Level 2 agent interaction trace

**LOADED FROM FROZEN 2025 FINAL-TEST ARTIFACTS - NOT RECOMPUTED.** Agents emit typed scores, confidence, fit/feature cutoffs and reason codes. Risk and execution remain separate from signal generation.


In [8]:
trace_rows = representative_trace_rows(ctx)
print(
    markdown_table(
        trace_rows,
        ["agent", "symbol", "score", "confidence", "fit_cutoff", "feature_cutoff", "reason_codes"],
    )
)

| agent | symbol | score | confidence | fit_cutoff | feature_cutoff | reason_codes |
| --- | --- | --- | --- | --- | --- | --- |
| sma_crossover | BTC/USDT | -1.000 | 0.228 | 2024-12-31T00:00:00+00:00 | 2025-01-01T00:00:00+00:00 | ok |
| econometric_ar_garch | BTC/USDT | 0.037 | 0.049 | 2024-12-31T00:00:00+00:00 | 2025-01-01T00:00:00+00:00 | ok |
| ml_logistic | BTC/USDT | -0.128 | 0.128 | 2024-12-31T00:00:00+00:00 | 2025-01-01T00:00:00+00:00 | ok |
| ml_hist_gradient_boosting | BTC/USDT | -0.236 | 0.236 | 2024-12-31T00:00:00+00:00 | 2025-01-01T00:00:00+00:00 | ok |
| aggregator | BTC/USDT | -0.466 | 0.040 | 2024-12-31T00:00:00+00:00 | 2025-01-01T00:00:00+00:00 | ok |
| post_risk | portfolio | approve | cash=100.0% | 2025-01-01T00:00:00+00:00 | 2025-01-01T00:00:00+00:00 | ok |


## 8. Level 1 - Baseline Strategy for a Single Cryptocurrency

**LOADED FROM FROZEN 2025 FINAL-TEST ARTIFACTS - NOT RECOMPUTED.** BTC/USDT SMA baseline through the shared next-open engine.


In [9]:
frame = ctx.metrics["level_1"]
important = [
    "approach",
    "method",
    "policy",
    "selected_for_level_1",
    "net_total_return",
    "net_sharpe",
    "net_max_drawdown",
    "net_turnover",
    "net_total_cost",
    "gross_total_return",
    "net_benchmark_total_return",
    "eligible_count_max",
    "scored_count_max",
    "selected_count_max",
    "rebalance_submitted_count",
    "runtime_seconds",
    "peak_rss_mb",
]
available = [column for column in important if column in frame.columns]
print(frame[available].to_string(index=False))

 net_total_return  net_sharpe  net_max_drawdown  net_turnover  net_total_cost  gross_total_return  net_benchmark_total_return
        -0.074388   -0.171018         -0.184693      5.987459     8906.270173           -0.067448                   -0.054209


## 9. Level 2 - AI Agents, Econometrics and ML

**LOADED FROM FROZEN 2025 FINAL-TEST ARTIFACTS - NOT RECOMPUTED.** Single-pair SMA, AR/GARCH, ML and ensemble candidates under identical costs and timing.


In [10]:
frame = ctx.metrics["level_2"]
important = [
    "approach",
    "method",
    "policy",
    "selected_for_level_2",
    "net_total_return",
    "net_sharpe",
    "net_max_drawdown",
    "net_turnover",
    "net_total_cost",
    "gross_total_return",
    "net_benchmark_total_return",
    "eligible_count_max",
    "scored_count_max",
    "selected_count_max",
    "rebalance_submitted_count",
    "runtime_seconds",
    "peak_rss_mb",
]
available = [column for column in important if column in frame.columns]
print(frame[available].to_string(index=False))

                 approach  selected_for_level_2  net_total_return  net_sharpe  net_max_drawdown  net_turnover  net_total_cost  gross_total_return  net_benchmark_total_return
            technical_sma                 False         -0.079399   -0.382857         -0.188766      9.156109    13915.322654           -0.066666                   -0.054209
     econometric_ar_garch                 False         -0.036786   -1.410950         -0.040814     20.337617    29710.764234           -0.006946                   -0.054209
              ml_logistic                 False         -0.003582   -0.601837         -0.005907      1.936327     2900.237881           -0.000682                   -0.054209
ml_hist_gradient_boosting                 False          0.000204    0.020397         -0.012807      4.623204     6946.829905            0.007163                   -0.054209
           agent_ensemble                  True         -0.005983   -0.522496         -0.013753      1.065023     1599.949191     

## 10. Level 3 - Static Portfolio Management on Historical Data

**LOADED FROM FROZEN 2025 FINAL-TEST ARTIFACTS - NOT RECOMPUTED.** 5-7 asset static portfolios estimated from the immediately prior 12 months.


In [11]:
frame = ctx.metrics["level_3"]
important = [
    "approach",
    "method",
    "policy",
    "selected_for_level_3",
    "net_total_return",
    "net_sharpe",
    "net_max_drawdown",
    "net_turnover",
    "net_total_cost",
    "gross_total_return",
    "net_benchmark_total_return",
    "eligible_count_max",
    "scored_count_max",
    "selected_count_max",
    "rebalance_submitted_count",
    "runtime_seconds",
    "peak_rss_mb",
]
available = [column for column in important if column in frame.columns]
print(frame[available].to_string(index=False))

            method  selected_for_level_3  net_total_return  net_sharpe  net_max_drawdown  net_turnover  net_total_cost  gross_total_return  net_benchmark_total_return
      equal_weight                 False         -0.254122   -0.124950         -0.487392         0.995          1492.5           -0.252630                   -0.254122
inverse_volatility                 False         -0.163871    0.001820         -0.447653         0.995          1492.5           -0.162378                   -0.254122
  minimum_variance                 False          0.008990    0.280430         -0.363158         0.995          1492.5            0.010482                   -0.254122
     cvar_downside                  True         -0.179797   -0.023384         -0.452392         0.995          1492.5           -0.178305                   -0.254122


## 11. Level 4 - Dynamic Portfolio Rebalancing

**LOADED FROM FROZEN 2025 FINAL-TEST ARTIFACTS - NOT RECOMPUTED.** Dynamic small-portfolio policies selected on validation with turnover constraints.


In [12]:
frame = ctx.metrics["level_4"]
important = [
    "approach",
    "method",
    "policy",
    "selected_for_level_4",
    "net_total_return",
    "net_sharpe",
    "net_max_drawdown",
    "net_turnover",
    "net_total_cost",
    "gross_total_return",
    "net_benchmark_total_return",
    "eligible_count_max",
    "scored_count_max",
    "selected_count_max",
    "rebalance_submitted_count",
    "runtime_seconds",
    "peak_rss_mb",
]
available = [column for column in important if column in frame.columns]
print(frame[available].to_string(index=False))

                 policy  selected_for_level_4  net_total_return  net_sharpe  net_max_drawdown  net_turnover  net_total_cost  gross_total_return  net_benchmark_total_return  rebalance_submitted_count
static_level3_benchmark                 False         -0.179797   -0.023384         -0.452392      0.995000     1492.500000           -0.178305                   -0.179797                        1.0
       calendar_monthly                  True         -0.040515   -0.880172         -0.091319      2.372997     3584.463908           -0.037076                   -0.093305                      143.0
          drift_monthly                 False          0.038498    0.290045         -0.234626      8.389893    12733.069535            0.051690                   -0.093305                      156.0
    signal_risk_monthly                 False          0.038498    0.290045         -0.234626      8.389893    12733.069535            0.051690                   -0.093305                      156.0


## 12. Level 5 - Large-Universe Dynamic Portfolio Expansion

**LOADED FROM FROZEN 2025 FINAL-TEST ARTIFACTS - NOT RECOMPUTED.** Deterministic cross-sectional scoring over a 100+ point-in-time universe with top-K allocation.


In [13]:
frame = ctx.metrics["level_5"]
important = [
    "approach",
    "method",
    "policy",
    "selected_for_level_5",
    "net_total_return",
    "net_sharpe",
    "net_max_drawdown",
    "net_turnover",
    "net_total_cost",
    "gross_total_return",
    "net_benchmark_total_return",
    "eligible_count_max",
    "scored_count_max",
    "selected_count_max",
    "rebalance_submitted_count",
    "runtime_seconds",
    "peak_rss_mb",
]
available = [column for column in important if column in frame.columns]
print(frame[available].to_string(index=False))

 net_total_return  net_sharpe  net_max_drawdown  net_turnover  net_total_cost  gross_total_return  net_benchmark_total_return  eligible_count_max  scored_count_max  selected_count_max  rebalance_submitted_count  runtime_seconds  peak_rss_mb
          -0.2799     -0.2183         -0.421704     49.244675   110938.845706           -0.186222                     -0.0544               120.0             120.0                25.0                      364.0         75.24469   727.296875


## 13. Level 5 economic diagnosis

**LOADED FROM FROZEN 2025 FINAL-TEST ARTIFACTS - NOT RECOMPUTED.** Level 5 demonstrated scale, but not profitable economics. The deterministic score selected diversified top-K baskets; frequent changes generated high turnover and costs while gross performance was already negative.


In [14]:
rebalance = pd.read_parquet(FINAL_DIR / "monitoring/level_5_rebalance_log.parquet")
orders = pd.read_parquet(FINAL_DIR / "orders/level_5.parquet")
fills = pd.read_parquet(FINAL_DIR / "fills/level_5.parquet")
weights = pd.read_parquet(FINAL_DIR / "weights/level_5.parquet")
asset_cols = [column for column in weights.columns if "/" in column]
l5 = ctx.metrics["level_5"].iloc[0]
diag = [
    {
        "metric": "eligible/scored/selected max",
        "value": (
            f"{int(l5['eligible_count_max'])}/"
            f"{int(l5['scored_count_max'])}/"
            f"{int(l5['selected_count_max'])}"
        ),
    },
    {"metric": "submitted rebalances", "value": str(int(rebalance["submitted_to_broker"].sum()))},
    {"metric": "fills", "value": str(len(fills))},
    {"metric": "fee-bearing order notional", "value": money(orders["target_notional"].abs().sum())},
    {"metric": "total transaction costs", "value": money(fills["total_cost"].sum())},
    {"metric": "reported turnover", "value": format_float(l5["net_turnover"])},
    {"metric": "gross return", "value": pct(l5["gross_total_return"])},
    {"metric": "net return", "value": pct(l5["net_total_return"])},
    {
        "metric": "full-cash days",
        "value": str(int((weights[asset_cols].abs().sum(axis=1) < 1e-9).sum())),
    },
    {"metric": "average risky exposure", "value": pct(weights[asset_cols].sum(axis=1).mean())},
    {"metric": "average cash weight", "value": pct(weights["cash_weight"].mean())},
    {
        "metric": "average nonzero risky holdings",
        "value": format_float((weights[asset_cols].abs() > 1e-12).sum(axis=1).mean()),
    },
]
print(markdown_table(diag, ["metric", "value"]))
print("approval_actions:", rebalance["approval_action"].value_counts().to_dict())

| metric | value |
| --- | --- |
| eligible/scored/selected max | 120/120/25 |
| submitted rebalances | 364 |
| fills | 8727 |
| fee-bearing order notional | $73,959,230 |
| total transaction costs | $110,939 |
| reported turnover | 49.24 |
| gross return | -18.62% |
| net return | -27.99% |
| full-cash days | 52 |
| average risky exposure | 85.35% |
| average cash weight | 14.65% |
| average nonzero risky holdings | 21.44 |
approval_actions: {'approve': 312, 'cash': 52, 'prior_weights': 1}


## 14. Cross-level conclusion, limitations and no-live-trading boundary

Net after fees and slippage is primary. Final-test results did not establish robust alpha, and several validation-selected methods did not generalize cleanly. Level 2 reduced downside versus BTC in 2025, but the robustness interval is wide and the randomization p-value is not significant. Level 5 handled 120 pairs, scored 120 and selected 25, but turnover and costs dominated an already negative gross result.

Known limitations: active-market survivorship/delisting bias, daily-bar liquidity proxy, simplified fills, USDT cash assumption, long-only unlevered spot scope, BTC-normalized Level 5 benchmark, short late-December 2024 Level 5 validation proof window, cash-heavy risk behavior, and dirty Stage 11 runner provenance. Future multi-CEX, Telegram and news/sentiment integrations are roadmap items only; no live order submission is enabled.
